In [ ]:
using LinearAlgebra: tr, I
using Parameters
using StaticArrays, HCubature, QuadGK
import DifferentialEquations
using Interpolations
using NLsolve
using ForwardDiff: gradient

solve(f, x0) = nlsolve(f, x0; inplace=false, show_trace=true, method=:newton, ftol=1e-6, iterations=20)
saddle_points(f, x0) = solve(x -> gradient(f, x), x0)
∫dk(f; d=2,atol=1e-7,rtol=1e-5) = hcubature(f, repeat([-π], d), repeat([π], d); atol=atol, rtol=rtol)[1] / (2π)^d
∫dt(t0,tf,f;atol=1e-8,rtol=1e-6) = quadgk(f, t0, tf; atol=atol, rtol=rtol)[1]

In [ ]:
"""
Cubic Anderson model parameters (units of tc)
"""
@with_kw mutable struct PAM
  V₀::Float64 = 0.    # Hybridization strength
  tf::Float64 = 0.    # F-electron hopping
  ε₀f::Float64 = 0.   # F-electron ground state e
  μ::Float64 = 0.     # Chemical potential
  β::Float64 = Inf    # Inverse temperature

  b::Float64 = NaN       # Saddle-point condensed slave boson
  λ::Float64 = NaN       # Saddle-point lagrange multiplier
end

"""
Slave-boson mean-field free energy at T -> 0
"""
function FreeEnergy(x::Vector, p::PAM)
  b, λ = x

  function bands(k::SVector)
    c = -2.0 * sum(cos, k) - p.μ
    f = -2.0p.tf * sum(cos, k) * abs2(b) + λ - p.μ
    h = (p.V₀ * b)

    σ = 0.5 * (c + f)
    Δ = sqrt((0.5 * (c - f))^2 + abs2(h))

    return σ - Δ, σ - Δ, σ + Δ, σ + Δ
  end

  return (λ - p.ε₀f) * (abs2(b) - 1) + ∫dk(k -> sum(bands(k)[1:2]))
end

"""
Slave-boson mean-field occupation of the fermionic fields
"""
function G0L(k::SVector, p::PAM, b::Number=p.b, λ::Number=p.λ)
  c = -2.0 * sum(cos, k) - p.μ
  f = -2.0p.tf * sum(cos, k) * abs2(b) + λ - p.μ
  h = (p.V₀ * b)

  σ = 0.5 * (c + f)
  δ = 0.5 * (c - f)
  Δ = sqrt(δ^2 + abs2(h))
  return SMatrix{2,2,ComplexF64}(1 - δ/Δ, -h'/Δ, -h/Δ, 1 + δ/Δ)
end

"""
Slave-boson mean-field free energy at T -> 0
"""
function sbmf_equations(x::Vector, p::PAM)
  b, λ = x
  G0 = ∫dk(k-> G0L(k, p, b, λ))
  return [(λ - p.ε₀f) * b + p.V₀ * G0[2,1], abs2(b) + G0[2,2] - 1]
end

In [ ]:
# Parameters
pam_0 = PAM(V₀=1.0, tf=-0.0, ε₀f=-1.5)

# Find the fields that minimize the Free Energy
@show solution = saddle_points(x -> FreeEnergy(x, pam_0), [0.2,1e-1])
pam_0.b, pam_0.λ = solution.zero

In [ ]:
# Parameters
pam_0 = PAM(V₀=1.0, tf=-0.0, ε₀f=-1.5)

# Solve the slave-boson mean-field equations
@show solution = solve(x -> sbmf_equations(x, pam_0), [0.2,1e-1])
pam_0.b, pam_0.λ = solution.zero

In [ ]:
using Test

@testset "Initial conditions" begin
  G0L_ = ∫dk(k -> G0L(k, pam_0, pam_0.b, pam_0.λ))
  
  # λ * b + V cf = 0
  @test G0L_[1,2] * pam_0.V₀ + (pam_0.λ - pam_0.ε₀f) * pam_0.b ≈ 0.0 atol=1e-5
  
  # f^2 + b^2 = 1
  @test G0L_[2,2] + pam_0.b^2 ≈ 1.0 atol=1e-5
  
  # Half filling
  @test G0L_[1,1] + G0L_[2,2] ≈ 2
end

In [ ]:
# Photon occupation
function ξ(t; Ω=1,tc=5)
  return (Ω^2 / (2π))^(1/4) * exp(-(t - tc)^2 * (Ω^2 / 4))
end

const Γ = sqrt(0.2)

"""
Time-dependent slave-boson mean-field equations with photon coupling
"""
function f′(u, h, params::Tuple{PAM, Interpolations.GriddedInterpolation}, t::Float64)
  println("\tAt t: $t")
  
  p = params[1]
  λ = params[2]
  b(t′) = h(params, t′)
  
  # G<(t,t)
  function GL(k::SVector, bt::ComplexF64, b2t::ComplexF64, λt::ComplexF64)::SMatrix{2,2,ComplexF64}
    U = exp(-1.0im * SMatrix{2,2,ComplexF64}(
      (-2.0 * sum(cos, k) - p.μ) * t, p.V₀ * bt, 
      (p.V₀ * bt)', -2.0p.tf * sum(cos, k) * b2t + λt - p.μ * t))
    return U * G0L(k, p, b(0.), λ(0.)) * U'
  end

  # solve for λ: b^2 + <f^† f> = 1
  λ.coefs[end] = solve(λ′ -> begin
    λ.knots[1][end] = t
    λ.coefs[end] = λ′[1]

    x = ∫dt(0, t, t′ -> [b(t′), abs2(b(t′)), λ(t′)])
    z = ∫dk(k -> begin
      g = GL(k, x[1], x[2], x[3])
      α = (g[1,2] * b(t)' + g[2,1] * b(t))
      dαdλ = g[2,2] * α
      SVector{3,ComplexF64}(g[2,2], α, dαdλ)
    end)

    real(abs2(b(t)) + z[1] - 1 + 2Γ^2 * ξ(t) * z[2] * z[3])
  end, [λ.coefs[end-1]]).zero[1]
  
  # solve for b: (i ∂t b + λ b + <c^† f> = 0)
  x = ∫dt(0, t, t′ -> [b(t′), abs2(b(t′)), λ(t′)])
  z::SVector{3,ComplexF64} = ∫dk(k -> begin
    g = GL(k, x[1], x[2], x[3])
    α = (g[1,2] * b(t)' + g[2,1] * b(t))
    dαdb = g[1,2] + p.V₀ * (g[1,1] * g[2,2] * b(t)' + g[2,1] * g[2,1] * b(t))
    SVector{3,ComplexF64}(g[2,1], α, dαdb)
      end)
  return -1.0im * ((λ(t) - p.ε₀f) * b(t) + p.V₀ * z[1] + 2Γ^2 * ξ(t) * z[2] * z[3])
end

In [ ]:
# Because λ is found through an implicit equation (and not as the result of the time integration)
# we need to make some tricks to properly save and access this function at all `t`s

# In order to interpolate some function we need at least two points, hence we add an artificial past point
λ = interpolate(([-2.0, -1.0],), [pam_0.λ,pam_0.λ], Gridded(Linear()))

# Since the ODE solver is adaptive we should only store the points accepted by the integrator
# This can be done with a `SavingCallback` which will add a new node to the interpolator's data
cb = DifferentialEquations.SavingCallback((u,t,integrator)-> begin
  push!(λ.knots[1], t)
  push!(λ.coefs, NaN)
  u
  end, DifferentialEquations.SavedValues(Float64, ComplexF64))

# Time span
tspan = (0.0, 10.0)

# Initial condition
u0 = complex(pam_0.b)

# History functio (b(t) for t <0)
h(p,t) = complex(pam_0.b)

# Time-dependent PAM
pam_t = deepcopy(pam_0)

prob = DifferentialEquations.DDEProblem(f′, u0, h, tspan, p=(pam_t, λ))
sol = DifferentialEquations.solve(
  prob, DifferentialEquations.MethodOfSteps(DifferentialEquations.VCABM()); callback=cb)

In [ ]:
using Plots

plot(sol.t, abs2.(sol.u); label="b^2", legend=:topleft, xlabel="t", fmt = :png)
plot!(twinx(), sol.t, ξ.(sol.t), c=:orange, label="Photon pulse amplitude", legend=:topright)

# plot(cb.affect!.saved_values.t, abs2.(cb.affect!.saved_values.saveval))
# plot!(twinx(), cb.affect!.saved_values.t, ξ.(cb.affect!.saved_values.t), c=:orange, label="Photon pulse amplitude", legend=:topright)

# plot(sol.t, abs2.(λ.(sol.t)), label="λ^2")